<a href="https://colab.research.google.com/github/RaihahMahmud/FlyRank-AI--starter-ML-Internship-/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Finding 1 — The Anatomy of Growing Content

The paper compares pages with rising impressions against pages with falling impressions. Growing pages were, on average, longer, younger, and slightly better positioned. The paper reports 74,187 rising pages versus 45,272 falling pages and describes the result as an observational comparison.

**Methodology questions:**

* **Where does the label come from?**
  The growing/declining label comes from observed impression trends. I would need to check the exact time window and threshold used to define "rising" and "falling" before treating the label as a general outcome.

* **Does the validation design support the claim?**
  The comparison supports a descriptive association between trend direction and page characteristics. It does not show that being longer or younger causes a page to grow.

* **What could confound the finding?**
  Content age, baseline visibility, topic, and differences between clients could affect both page characteristics and performance. The paper itself notes that content age is an important confounding factor in model-performance comparisons.

**Safe interpretation:**
The evidence supports saying that growing and declining pages have different observed profiles in this dataset. It does not support claiming that changing one of these characteristics will cause growth.

---

### Finding 2 — The Freshness Multiplier

The paper identifies the 31–90 day window as the strongest stable freshness band, with a growth-to-decline ratio of 7.88:1. It also reports that 365+ day pages refreshed within 30 days had 3.2× higher Health Score and 57× more impressions than the comparison group. However, the paper warns that the 361+ freshness bucket is very small and unstable.

**Methodology questions:**

* **Where does the label come from?**
  The freshness groups are based on content age and the growth/decline comparison is based on observed performance. I would want the exact definition of "refreshed within 30 days" and the comparison window before treating refresh as an outcome-related label.

* **Does the validation design support the claim?**
  The reported comparisons show an association between freshness/refresh status and observed performance. They do not establish that freshness or refreshing a page caused the improvement.

* **Could there be a mechanical relationship with the Health Score?**
  Yes. The paper defines Health Score using impressions, position, CTR, and scroll depth. Therefore, comparing groups using Health Score can partly reflect differences in the same performance measurements used to construct the score.

* **What could confound the finding?**
  Older and newer pages may differ in topic, baseline performance, client, and other characteristics. Pages selected for refresh may also be systematically different from pages that were not refreshed.

**Safe interpretation:**
The evidence shows a strong observed relationship between freshness/refresh status and performance in this dataset. It does not prove that freshness is a causal "multiplier" or that refreshing a page will produce the reported improvement.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

==>My model under an honest split

My Week-5 model used a client-grouped train/test split, which keeps pages from the same client from appearing in both training and testing. The Week-5 result was very strong, with Logistic Regression achieving 0.9993 precision, 0.9863 recall, and 0.9927 F1 on the held-out test set.

For this validation audit, I make the evaluation stricter by performing the client-grouped split before calculating the target threshold and baseline thresholds. The thresholds are learned from the training clients only and then applied to the unseen test clients.

This avoids using information from the test clients when constructing the training target or baseline. The comparison therefore shows how the Week-5 result changes when the validation procedure is more careful about information flow.

I treat any difference between the two results as a validation finding, not as evidence that the model will improve the performance of refreshed content.


In [1]:
# Reloading March data from Hugging Face using the authenticated local download from previously for model in ML-08

import os
import duckdb
import pandas as pd
import numpy as np

from huggingface_hub import login, hf_hub_download
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing from Colab Secrets.")

login(token=HF_TOKEN, add_to_git_credential=False)

local_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

con = duckdb.connect()

march = con.sql(f"""
    SELECT *
    FROM read_parquet('{local_file}')
""").df()

print("March data loaded successfully")
print("March rows:", len(march))
print("March dates:", march["report_date"].nunique())
print("Columns:", len(march.columns))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March data loaded successfully
March rows: 9841378
March dates: 31
Columns: 31


In [2]:
##### ===> SECTION 2 — HONEST CLIENT-GROUPED VALIDATION
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

#  page/client-level data
m = march[
    ["client_hash_id", "content_hash_id",
     "gsc_impressions", "gsc_clicks", "gsc_avg_position"]
].copy()

m = m.dropna(subset=["gsc_impressions", "gsc_clicks"])

m["ctr"] = (
    m["gsc_clicks"] /
    m["gsc_impressions"].replace(0, np.nan)
).fillna(0)

base_honest = (
    m.groupby(["client_hash_id", "content_hash_id"], as_index=False)
     .agg(
         gsc_impressions=("gsc_impressions", "sum"),
         gsc_clicks=("gsc_clicks", "sum"),
         ctr=("ctr", "mean"),
         gsc_avg_position=("gsc_avg_position", "mean")
     )
)

feature_cols = ["gsc_impressions", "ctr", "gsc_avg_position"]

base_honest = base_honest.dropna(
    subset=feature_cols + ["client_hash_id"]
).reset_index(drop=True)


X = base_honest[feature_cols]
groups = base_honest["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, groups=groups)
)

train = base_honest.iloc[train_idx].copy()
test = base_honest.iloc[test_idx].copy()

# 2.Learning thresholds from training data
click_threshold = train["gsc_clicks"].median()
imp_threshold = train["gsc_impressions"].quantile(0.75)
ctr_threshold = train["ctr"].quantile(0.25)

train["target"] = (train["gsc_clicks"] > click_threshold).astype(int)
test["target"] = (test["gsc_clicks"] > click_threshold).astype(int)

train["baseline_prediction"] = (
    (train["gsc_impressions"] > imp_threshold) &
    (train["ctr"] < ctr_threshold)
).astype(int)

test["baseline_prediction"] = (
    (test["gsc_impressions"] > imp_threshold) &
    (test["ctr"] < ctr_threshold)
).astype(int)

# 3. Training WEEK-5 LOGISTIC REGRESSION
model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(train[feature_cols], train["target"])

prob = model.predict_proba(test[feature_cols])[:, 1]
pred = (prob >= 0.5).astype(int)

# 4. COMParing
comparison_honest = pd.DataFrame([
    {
        "Method": "Week-5 reported result",
        "Precision": 0.9993,
        "Recall": 0.9863,
        "F1": 0.9927
    },
    {
        "Method": "Honest client-grouped rerun",
        "Precision": precision_score(test["target"], pred, zero_division=0),
        "Recall": recall_score(test["target"], pred, zero_division=0),
        "F1": f1_score(test["target"], pred, zero_division=0)
    },
    {
        "Method": "Honest baseline",
        "Precision": precision_score(
            test["target"], test["baseline_prediction"], zero_division=0
        ),
        "Recall": recall_score(
            test["target"], test["baseline_prediction"], zero_division=0
        ),
        "F1": f1_score(
            test["target"], test["baseline_prediction"], zero_division=0
        )
    }
])

print("HONEST CLIENT-GROUPED VALIDATION")

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train clients:", train["client_hash_id"].nunique())
print("Test clients:", test["client_hash_id"].nunique())
print("Client overlap:",
      len(set(train["client_hash_id"]) & set(test["client_hash_id"])))

display(comparison_honest.round(4))

HONEST CLIENT-GROUPED VALIDATION
Train rows: 138310
Test rows: 38428
Train clients: 37
Test clients: 10
Client overlap: 0


,Method,Precision,Recall,F1
0,Week-5 reported result,0.9993,0.9863,0.9927
1,Honest client-grouped rerun,0.9993,0.9863,0.9927
2,Honest baseline,0.0000,0.0000,0.0000


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Feature audit

The Week-5 model uses three features:

- `gsc_impressions`
- `ctr`
- `gsc_avg_position`

These are observed search-performance signals from the March 2026 data. No future-window features were used.

### Validation audit

The train and test sets have **zero client overlap**, so the model was evaluated on clients that were not present in training.

For the honest rerun, the target and baseline thresholds were calculated using the training data only. The test data was not used to learn these thresholds.

### Conclusion

The leakage audit found:

- No future-window features were used.
- No client overlap exists between train and test.
- No thresholds were learned from test data.

The corrected validation procedure therefore provides a more conservative evaluation of the Week-5 model.

In [4]:
print("LEAKAGE AUDIT")

print("Model features:", feature_cols)
print("Future-window features used: NONE")
print("Client overlap between train/test:",
      len(set(train["client_hash_id"]) & set(test["client_hash_id"])))
print("Thresholds learned from test data: NO")

LEAKAGE AUDIT
Model features: ['gsc_impressions', 'ctr', 'gsc_avg_position']
Future-window features used: NONE
Client overlap between train/test: 0
Thresholds learned from test data: NO


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

==>
### Bold claim

> The model predicts which pages should be refreshed to improve their Google performance.

### Safer claim

> The model identifies and ranks content pages based on observed search-performance signals and measured click-volume patterns in held-out client data.

This is a **directional, decision-support signal** for prioritizing pages for content review. It does not show that refreshing a page will cause higher clicks, impressions, CTR, or rankings.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.